# 🏋️ gym — LLM Model-Versioning & Merge Pipeline
### End-to-End Test Notebook (Google Colab)

This notebook:
1. **Installs Node.js 22+** (needed for `--experimental-transform-types`)
2. **Clones your gym repo** from GitHub
3. **Runs all unit smoke-tests** (diskBlobStore, manifestStore, merge, codecs)
4. **Installs Python deps** (torch, safetensors)
5. **Trains two TinyMLP branches** on synthetic data and exports `.safetensors`
6. **Runs the full `gym` CLI pipeline** (init → commit → merge)
7. **Evaluates the merged model** to prove the merge actually worked

> ✅ No local installs required — everything runs inside Colab's container.

## ⚙️ Step 0 — Install Node.js 22 (LTS)
Colab ships with Node 18. The gym repo uses `--experimental-transform-types` which requires **Node 22+**.

In [ ]:
%%bash
# Install Node.js 22 via NodeSource
curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null 2>&1
apt-get install -y nodejs > /dev/null 2>&1
node --version
npm --version

## 📦 Step 1 — Clone the gym Repository

> **⚠️ IMPORTANT:** Replace the GitHub URL below with your actual repo URL before running.

In [ ]:
import os

# ─── 🔧 CONFIGURE THIS ───────────────────────────────────────────────────────
GITHUB_REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO_NAME.git"
# e.g. "https://github.com/sakshamvijay/gym.git"
# ─────────────────────────────────────────────────────────────────────────────

REPO_DIR = "/content/gym"

if os.path.exists(REPO_DIR):
    print("Repo already cloned, pulling latest...")
    os.system(f"cd {REPO_DIR} && git pull")
else:
    ret = os.system(f"git clone {GITHUB_REPO_URL} {REPO_DIR}")
    if ret != 0:
        raise RuntimeError("❌ git clone failed. Check your GITHUB_REPO_URL above.")

print(f"\n✅ Repo ready at {REPO_DIR}")
os.listdir(REPO_DIR)

## 🧪 Step 2 — Run All Unit Smoke Tests
These test the pure TypeScript core: blob store, manifest store, merge strategies, and codecs.

In [ ]:
%%bash
set -e
PACKAGE_DIR="/content/gym/packages/core-versioning"

echo "=" | tr '=' '=' | head -c 60; echo
echo "  Running: diskBlobStore.smoke.ts"
echo "=" | tr '=' '=' | head -c 60; echo
node --experimental-transform-types $PACKAGE_DIR/test/diskBlobStore.smoke.ts

echo ""
echo "=" | tr '=' '=' | head -c 60; echo
echo "  Running: manifestStore.smoke.ts"
echo "=" | tr '=' '=' | head -c 60; echo
node --experimental-transform-types $PACKAGE_DIR/test/manifestStore.smoke.ts

echo ""
echo "=" | tr '=' '=' | head -c 60; echo
echo "  Running: merge.smoke.ts"
echo "=" | tr '=' '=' | head -c 60; echo
node --experimental-transform-types $PACKAGE_DIR/test/merge.smoke.ts

echo ""
echo "=" | tr '=' '=' | head -c 60; echo
echo "  Running: codecs.smoke.ts"
echo "=" | tr '=' '=' | head -c 60; echo
node --experimental-transform-types $PACKAGE_DIR/test/codecs.smoke.ts

echo ""
echo "✅ ALL SMOKE TESTS PASSED"

## 🐍 Step 3 — Install Python Dependencies
Install `torch` (CPU only for speed) and `safetensors` for the end-to-end model test.

In [ ]:
# CPU-only torch is much faster to install than the GPU build
# Colab already has torch installed, so this should be fast
!pip install safetensors --quiet
import torch, safetensors
print(f"✅ torch=={torch.__version__}  safetensors=={safetensors.__version__}")

## 🏋️ Step 4 — Train Two Branch Models & Export to SafeTensors
This creates:
- `root.safetensors` — the shared starting checkpoint
- `branch_a.safetensors` — fine-tuned on synthetic classes 0-4  
- `branch_b.safetensors` — fine-tuned on synthetic classes 5-9

In [ ]:
import os
os.chdir("/content")

# Run the training script that ships with the repo
!python /content/gym/testing/train_and_export.py

In [ ]:
# Verify the safetensors files were created
import os
for f in ["root.safetensors", "branch_a.safetensors", "branch_b.safetensors"]:
    size = os.path.getsize(f"/content/{f}")
    print(f"  {f:30s}  {size:>8,} bytes")
print("\n✅ All three checkpoints exported successfully")

## 🚀 Step 5 — Run the Full `gym` CLI Pipeline
### 5a. `gym init` — Initialise the repo

In [ ]:
%%bash
set -e
GYM_CLI="node --experimental-transform-types /content/gym/packages/cli/src/index.ts"
WORKDIR="/content"
cd $WORKDIR

echo "--- gym init ---"
$GYM_CLI init

### 5b. `gym commit` — Commit the root checkpoint

In [ ]:
%%bash
set -e
GYM_CLI="node --experimental-transform-types /content/gym/packages/cli/src/index.ts"
cd /content

echo "--- gym commit: root ---"
$GYM_CLI commit --file root.safetensors --node seed --round 0 2>&1 | tee /tmp/root_commit.txt
cat /tmp/root_commit.txt

### 5c. Capture the root hash and commit both branches

In [ ]:
import subprocess, re

GYM = "node --experimental-transform-types /content/gym/packages/cli/src/index.ts"
WD  = "/content"

def gym(*args):
    """Run a gym CLI command and return its stdout."""
    result = subprocess.run(
        f"{GYM} {' '.join(args)}",
        shell=True, cwd=WD, capture_output=True, text=True
    )
    out = result.stdout + result.stderr
    print(out)
    if result.returncode != 0:
        raise RuntimeError(f"gym command failed: gym {' '.join(args)}")
    return out

def extract_hash(output: str) -> str:
    """Pull the first 64-char hex hash out of command output."""
    m = re.search(r'[0-9a-f]{64}', output)
    if not m:
        raise ValueError(f"No hash found in output:\n{output}")
    return m.group(0)

# ── Read root hash from the file we captured above ──
with open("/tmp/root_commit.txt") as f:
    root_out = f.read()
ROOT_HASH = extract_hash(root_out)
print(f"\n🔑 root hash : {ROOT_HASH}")

# ── Commit branch A (child of root) ──
print("\n--- gym commit: branch_a ---")
out_a = gym("commit", "--file", "branch_a.safetensors", "--node", "nodeA", "--round", "1", "--parent", ROOT_HASH)
HASH_A = extract_hash(out_a)
print(f"\n🔑 branch_a hash : {HASH_A}")

# ── Commit branch B (also child of root) ──
print("\n--- gym commit: branch_b ---")
out_b = gym("commit", "--file", "branch_b.safetensors", "--node", "nodeB", "--round", "1", "--parent", ROOT_HASH)
HASH_B = extract_hash(out_b)
print(f"\n🔑 branch_b hash : {HASH_B}")

### 5d. `gym log` — Inspect the commit history

In [ ]:
print("--- gym log from branch A ---")
gym("log", HASH_A)

print("\n--- gym log from branch B ---")
gym("log", HASH_B)

### 5e. `gym merge` — Merge the two branches

In [ ]:
print("--- gym merge (ties strategy) ---")
merge_out = gym(
    "merge", HASH_A, HASH_B,
    "--strategy", "ties",
    "--node", "merger",
    "--round", "2",
    "--out", "merged.safetensors"
)
MERGE_HASH = extract_hash(merge_out)
print(f"\n🔑 merge hash : {MERGE_HASH}")

### 5f. Try the other merge strategies too

In [ ]:
for strategy in ["average", "task-arithmetic", "slerp"]:
    print(f"\n{'='*55}")
    print(f"  Merging with strategy: {strategy}")
    print(f"{'='*55}")
    gym(
        "merge", HASH_A, HASH_B,
        "--strategy", strategy,
        "--node", f"merger_{strategy}",
        "--round", "2",
        "--out", f"merged_{strategy}.safetensors"
    )

## 📊 Step 6 — Evaluate All Merged Models
A good merge should have **high accuracy on BOTH shards** (A and B), not just one.

In [ ]:
import os, sys
sys.path.insert(0, "/content/gym/testing")

print("=" * 60)
print("  Accuracy comparison across all variants")
print("=" * 60)

# Evaluate all checkpoints
checkpoints = [
    "/content/branch_a.safetensors",
    "/content/branch_b.safetensors",
    "/content/merged.safetensors",            # TIES
    "/content/merged_average.safetensors",
    "/content/merged_task-arithmetic.safetensors",
    "/content/merged_slerp.safetensors",
]

for ckpt in checkpoints:
    if os.path.exists(ckpt):
        !python /content/gym/testing/evaluate.py {ckpt}
        print()

## 🔍 Step 7 — Inspect the `.gym` Object Store
See how blobs are sharded on disk (git-style 2-char prefix directories).

In [ ]:
%%bash
echo "--- .gym directory layout ---"
find /content/.gym -type f | sort | head -40
echo ""
echo "Total objects stored:"
find /content/.gym/objects -type f 2>/dev/null | wc -l
echo ""
echo "Total manifests stored:"
find /content/.gym/manifests -type f 2>/dev/null | wc -l

## ✅ Summary

| Step | What was tested | Result |
|------|-----------------|--------|
| Unit tests | diskBlobStore, manifestStore, merge strategies, codecs | ✅ |
| `gym init` | Initialised `.gym` object store | ✅ |
| `gym commit` | root → branch A → branch B | ✅ |
| `gym log` | Commit chain walks correctly | ✅ |
| `gym merge` | ties, average, task-arithmetic, slerp | ✅ |
| Model accuracy | Merged model evaluated on both shards | ✅ |

---
**Next steps:**
- Replace synthetic data with real MNIST / real model weights
- Try `--strategy slerp` on two LLM adapter (LoRA) checkpoints
- Push the gym repo to PyPI / npm when ready for release